In [4]:
import joblib
from pathlib import Path

In [ ]:
DATA_PATH = Path.cwd().parent / "data" / "interim"

X_train = joblib.load(DATA_PATH / "X_train.pkl")
X_test = joblib.load(DATA_PATH / "X_test.pkl")

# Evidently

In [2]:
!uv add evidently

Resolved 171 packages in 0.51ms
Audited 167 packages in 0.06ms


In [ ]:
from evidently import Report
from evidently.metrics import DriftedColumnsCount

# Crear y ejecutar el reporte de drift
report = Report(metrics = [
    DriftedColumnsCount(method="psi"),
])

drift_report = report.run(reference_data=X_train, current_data=X_test)
drift_report.dict()

{'metrics': [{'id': '389d65fb2d00b0140fed36faa82e6dc3',
   'metric_name': 'DriftedColumnsCount(drift_share=0.5,method=psi)',
   'config': {'type': 'evidently:metric_v2:DriftedColumnsCount',
    'drift_share': 0.5,
    'method': 'psi'},
   'value': {'count': 0.0, 'share': 0.0}}],
 'tests': []}

In [22]:
drifted_columns_count = int(drift_report.dict()["metrics"][0]["value"]["count"])

# Cargar datos a InfluxDB

In [26]:
db_params = {
    "db": "evidently_metrics",
    "u": "admin",
    "p": "admin",
    "precision": "ns"
}
headers = {
    "Content-Type": "text/plain; charset=utf-8",
}

BASE_URL = "http://localhost:8086"

In [17]:
import requests

r = requests.post(f"{BASE_URL}/query", params=db_params, data={"q": "SHOW DATABASES"})
r.json()

{'results': [{'statement_id': 0,
   'series': [{'name': 'databases',
     'columns': ['name'],
     'values': [['evidently_metrics'], ['_internal']]}]}]}

In [ ]:
from datetime import datetime, timedelta
import numpy as np

# Create InfluxDB line protocol string
DATA_POINTS = 100
drift_data = []

for i in range(DATA_POINTS):
    timestamp = datetime.now() - timedelta(hours=100) + timedelta(hours=i)
    line = f"drift_metrics drifted_columns_count={drifted_columns_count + np.random.randint(10)} {int(timestamp.timestamp()) * 1000000000}"
    drift_data.append(line)
drift_data[:5] # Mostrar las primeras 5 líneas

['drift_metrics drifted_columns_count=8 1762717476000000000',
 'drift_metrics drifted_columns_count=8 1762721076000000000',
 'drift_metrics drifted_columns_count=0 1762724676000000000',
 'drift_metrics drifted_columns_count=4 1762728276000000000',
 'drift_metrics drifted_columns_count=2 1762731876000000000']

In [25]:
payload = '\n'.join(drift_data)
payload[:200]

'drift_metrics drifted_columns_count=8 1762717476000000000\ndrift_metrics drifted_columns_count=8 1762721076000000000\ndrift_metrics drifted_columns_count=0 1762724676000000000\ndrift_metrics drifted_colu'

In [27]:
write_r = requests.post(f"{BASE_URL}/write", params=db_params,  data=payload, headers=headers)
write_r

<Response [204]>

# Dashboard en Grafana

![data_drift](../experiments/data_drift.png)